# GR5069:Homework 5: model deployment
## *Xuejing Yan*


In [0]:
dbutils.library.restartPython()

In [0]:
import pandas as pd
import numpy as np
import mlflow
import mlflow.sklearn
import matplotlib.pyplot as plt
import seaborn as sns

from pyspark.sql.functions import col, avg, round, count, when

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix


In [0]:
df_results  = spark.read.csv("/Volumes/gr5069/raw/f1_data/results.csv", header=True)
df_pitstops = spark.read.csv("/Volumes/gr5069/raw/f1_data/pit_stops.csv", header=True)
df_races    = spark.read.csv("/Volumes/gr5069/raw/f1_data/races.csv", header=True)
df_qualifying = spark.read.csv("/Volumes/gr5069/raw/f1_data/qualifying.csv", header=True)

In [0]:
results_clean = (
    df_results
    .filter(col("position").isNotNull())
    .filter(col("position") != "\\N") 
    .withColumn("raceId", col("raceId").cast("int"))
    .withColumn("driverId", col("driverId").cast("int"))
    .withColumn("constructorId", col("constructorId").cast("int"))
    .withColumn("grid", col("grid").cast("int"))
    .withColumn("laps", col("laps").cast("int"))
    .withColumn("position", col("position").cast("int"))
)

In [0]:
races_clean = (
    df_races
    .select("raceId", "year", "round")
    .withColumn("raceId", col("raceId").cast("int"))
    .withColumn("year", col("year").cast("int"))
)

In [0]:
pit_clean = (
    df_pitstops
    .withColumn("raceId", col("raceId").cast("int"))
    .withColumn("driverId", col("driverId").cast("int"))
    .withColumn("milliseconds", col("milliseconds").cast("double"))
    .groupBy("raceId", "driverId")
    .agg(
        round(avg("milliseconds"), 2).alias("avg_pit_ms"),
        count("*").alias("pit_stop_count")
    )
)

In [0]:
races_clean = (
    df_races
    .select("raceId", "year")
    .withColumn("raceId", col("raceId").cast("int"))
    .withColumn("year", col("year").cast("int"))
)

In [0]:
qualifying_clean = (
    df_qualifying
    .select("raceId", "driverId", "position")
    .withColumn("raceId", col("raceId").cast("int"))
    .withColumn("driverId", col("driverId").cast("int"))
    .withColumn("quali_position", col("position").cast("int"))
    .drop("position")
)

In [0]:
# Build modeling dataset
model_df = (
    results_clean
    .select("raceId", "driverId", "constructorId", "grid", "laps", "position")
    .join(pit_clean, on=["raceId", "driverId"], how="inner")
    .join(races_clean, on="raceId", how="left")
    .join(qualifying_clean, on=["raceId", "driverId"], how="left")
    .withColumn("podium", when(col("position") <= 3, 1).otherwise(0))
    .na.drop()
)

display(model_df)

In [0]:
# Convert to Pandas for sklearn models
df = model_df.toPandas()

### Model 1: Logistic Regression

In [0]:
feature_cols_1 = [
    "grid",
    "quali_position",
    "laps",
    "constructorId",
    "avg_pit_ms",
    "pit_stop_count",
    "year"
]

X = df[feature_cols_1]
y = df["podium"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)


In [0]:
def log_logistic(run_name, params, feature_cols, X_train, X_test, y_train, y_test):
    import mlflow.sklearn
    import tempfile
    import matplotlib.pyplot as plt
    import pandas as pd

    from sklearn.linear_model import LogisticRegression
    from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

    with mlflow.start_run(run_name=run_name) as run:
        
        # Create model, train it, and create predictions
        model = LogisticRegression(**params)
        model.fit(X_train, y_train)
        predictions = model.predict(X_test)
        probabilities = model.predict_proba(X_test)[:, 1]

        # Log model
        mlflow.sklearn.log_model(model, "logistic-regression-model")

        # Log params
        for param, value in params.items():
            mlflow.log_param(param, value)

        # Create metrics
        accuracy = accuracy_score(y_test, predictions)
        precision = precision_score(y_test, predictions, zero_division=0)
        recall = recall_score(y_test, predictions, zero_division=0)
        f1 = f1_score(y_test, predictions, zero_division=0)

        print("accuracy: {}".format(accuracy))
        print("precision: {}".format(precision))
        print("recall: {}".format(recall))
        print("f1: {}".format(f1))

        # Log metrics
        mlflow.log_metric("accuracy", accuracy)
        mlflow.log_metric("precision", precision)
        mlflow.log_metric("recall", recall)
        mlflow.log_metric("f1", f1)

        # Create feature importance
        coef_df = pd.DataFrame(
            list(zip(feature_cols, model.coef_[0])),
            columns=["Feature", "Coefficient"]
        ).sort_values("Coefficient", ascending=False)

        # Log importances using a temporary file
        temp = tempfile.NamedTemporaryFile(prefix="coefficients-", suffix=".csv")
        temp_name = temp.name
        try:
            coef_df.to_csv(temp_name, index=False)
            mlflow.log_artifact(temp_name, "coefficients")
        finally:
            temp.close()

        cm = confusion_matrix(y_test, predictions)

        # Create plot
        fig, ax = plt.subplots()
        ax.imshow(cm)
        ax.set_title("Confusion Matrix")
        ax.set_xlabel("Predicted")
        ax.set_ylabel("Actual")

        for i in range(cm.shape[0]):
            for j in range(cm.shape[1]):
                ax.text(j, i, cm[i, j], ha="center", va="center")

        # Log confusion matrix using a temporary file
        temp = tempfile.NamedTemporaryFile(prefix="confusion-matrix-", suffix=".png")
        temp_name = temp.name
        try:
            fig.savefig(temp_name)
            mlflow.log_artifact(temp_name, "confusion-matrix")
        finally:
            temp.close()

        return model, predictions, probabilities, run.info.run_id

In [0]:
params_lr = {
    "max_iter": 1000,
    "class_weight": "balanced",
    "solver": "liblinear",
    "random_state": 42
}


model1, y_pred, y_prob, run_id_1 = log_logistic(
    "HW5_Model1_Logistic_Podium",
    params_lr,
    feature_cols_1,
    X_train,
    X_test,
    y_train,
    y_test
)

In [0]:
testPDF1 = X_test.copy()

testPDF1["predicted_podium"] = y_pred

predDF_final1 = spark.createDataFrame(testPDF1)

display(predDF_final1)

In [0]:
predDF_final1.write.mode('overwrite').csv('/Volumes/gr5069/xy2723/exercise/model1_pred.csv')

In [0]:
# model 2 feature
feature_cols_2 = [
    "grid",
    "quali_position",
    "laps",
    "constructorId",
    "pit_stop_count",
    "year"
]

X2 = df[feature_cols_2]
y2 = df["avg_pit_ms"]

X2_train, X2_test, y2_train, y2_test = train_test_split(
    X2, y2, test_size=0.3, random_state=42)

In [0]:
# Model 2: Random Forest Regression

def log_rf_regression(run_name, params, feature_cols, X_train, X_test, y_train, y_test):

    import mlflow.sklearn
    import tempfile
    import matplotlib.pyplot as plt
    import pandas as pd
    import numpy as np
    from sklearn.ensemble import RandomForestRegressor
    from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

    with mlflow.start_run(run_name=run_name) as run:

        model = RandomForestRegressor(**params)
        model.fit(X_train, y_train)

        predictions = model.predict(X_test)

        mlflow.sklearn.log_model(model, "random-forest-regression-model")

        for param, value in params.items():
            mlflow.log_param(param, value)

        mae = mean_absolute_error(y_test, predictions)
        mse = mean_squared_error(y_test, predictions)
        rmse = np.sqrt(mse)
        r2 = r2_score(y_test, predictions)

        print("mae: {}".format(mae))
        print("mse: {}".format(mse))
        print("rmse: {}".format(rmse))
        print("r2: {}".format(r2))

        mlflow.log_metric("mae", mae)
        mlflow.log_metric("mse", mse)
        mlflow.log_metric("rmse", rmse)
        mlflow.log_metric("r2", r2)

        importance = pd.DataFrame(
            list(zip(feature_cols, model.feature_importances_)),
            columns=["Feature", "Importance"]
        ).sort_values("Importance", ascending=False)

        temp = tempfile.NamedTemporaryFile(prefix="feature-importance-", suffix=".csv")
        temp_name = temp.name
        try:
            importance.to_csv(temp_name, index=False)
            mlflow.log_artifact(temp_name, "feature-importance")
        finally:
            temp.close()

        fig, ax = plt.subplots()

        ax.scatter(y_test, predictions, alpha=0.5)
        ax.set_xlabel("Actual Average Pit Stop Time")
        ax.set_ylabel("Predicted Average Pit Stop Time")
        ax.set_title("Actual vs Predicted Pit Stop Time")

        temp = tempfile.NamedTemporaryFile(prefix="actual-vs-predicted-", suffix=".png")

        temp_name = temp.name

        try:
            fig.savefig(temp_name)
            mlflow.log_artifact(temp_name, "actual-vs-predicted")
        finally:
            temp.close()

        return model, predictions, run.info.run_id

In [0]:
params_rf = {
    "n_estimators": 100,
    "max_depth": 8,
    "max_features": "sqrt",
    "random_state": 22
}

model2, y2_pred, run_id_2 = log_rf_regression(
    "Model2_Random_Forest",
    params_rf,
    feature_cols_2,
    X2_train,
    X2_test,
    y2_train,
    y2_test
)

In [0]:
testPDF2 = X2_test.copy()
testPDF2["predicted_avg_pit_ms"] = y2_pred
predDF_final2 = spark.createDataFrame(testPDF2)

display(predDF_final2)

In [0]:
predDF_final2.write.csv('/Volumes/gr5069/xy2723/exercise/model2_pred.csv')